# Feature Track 3: Structured Outputs & Claim Labelling

---

The baseline RAG returns plain prose. For sustainability reporting this creates two problems:

1. **No machine-readable quality signal.** A sentence like *"The Logypal 1 has a GWP of 3.2 kg CO₂e"* looks identical whether the figure is third-party verified or a supplier self-declaration. A downstream compliance system cannot tell them apart.

2. **Claim laundering.** Unverified supplier self-declarations are presented in the same confident tone as independently audited figures.

## What Structured Outputs Fix

Forcing the LLM to return a **typed, validated data structure** instead of free text solves both problems:

| Field | Type | Purpose |
|---|---|---|
| `answer` | `str` | The plain-language answer |
| `evidence_level` | `VERIFIED / CLAIMED / MISSING / MIXED` | Machine-readable quality of the underlying data |
| `sources_used` | `list[str]` | Which documents the answer draws from |
| `gaps` | `list[str]` | What information is absent or unclear |
| `risk` | `LOW / MEDIUM / HIGH` | Risk of citing this answer in customer-facing communication |

## Three Implementation Approaches

| # | Approach | Reliability |
|---|---|---|
| A | **Prompt-only JSON coercion** | Low: LLM may add Markdown fences or prose around the JSON | 
| B | **OpenAI JSON mode** (`response_format: json_object`) | High: API guarantees syntactically valid JSON | 
| C | **Pydantic validation layer** | Adds type-safety on top of A or B: catches wrong enum values and missing fields |

 **Evidence levels** used throughout:

| Level | Meaning |
|---|---|
| `VERIFIED` | Third-party EPD or independent audit |
| `CLAIMED` | Supplier self-declaration, not independently verified |
| `MISSING` | Not found in any source document |
| `MIXED` | Some verified, some only claimed |

---

## Setup

**Prerequisites:** `conversational-toolkit` and `backend` installed in editable mode. Set `OPENAI_API_KEY` in your environment.

In [ ]:
import json
import re
from enum import Enum
from textwrap import dedent

from pydantic import BaseModel, Field, ValidationError

from conversational_toolkit.agents.base import QueryWithContext
from conversational_toolkit.agents.rag import RAG
from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    EMBEDDING_MODEL,
    VS_PATH,
    build_llm,
    load_chunks,
    build_vector_store,
)

# ---------------------------------------------------------------------------
# System prompts
# ---------------------------------------------------------------------------

SYSTEM_PROMPT_BASELINE = (
    "You are a helpful AI assistant specialised in sustainability and product compliance. "
    "Answer questions in one sentence using the provided sources. "
    "If the information is not in the sources, say so clearly."
)

SYSTEM_PROMPT_ENTITY_GROUNDED = dedent("""
    You are a sustainability compliance assistant for PrimePack AG. Answer questions using ONLY the provided sources.

    RULES (apply in order):
    1. Identify the key entity in the question (product name, supplier, product ID).
    2. Check that this exact entity appears in the retrieved sources.
       If it does NOT appear, respond: "The sources do not contain information about
       [entity]. I cannot answer this question." Do not substitute other products.
    3. Distinguish clearly between:
       VERIFIED — backed by a third-party EPD or independent audit
       CLAIMED  — supplier self-declaration, not independently verified
       MISSING  — not found in sources
    4. Label forward-looking targets (e.g. "carbon neutral by 2025") as targets,
       not as current verified status.
""").strip()

SYSTEM_PROMPT_STRUCTURED = dedent("""
    You are a sustainability compliance assistant for PrimePack AG.
    Answer questions using ONLY the provided sources.

    Always respond with a JSON object matching this schema:
    {
      "answer": "<plain language answer>",
      "evidence_level": "<VERIFIED | CLAIMED | MISSING | MIXED>",
      "sources_used": ["<source file 1>", ...],
      "gaps": ["<missing or unclear information>", ...],
      "risk": "<LOW | MEDIUM | HIGH — risk of citing this in customer communication>"
    }

    Evidence levels:
      VERIFIED — third-party EPD or independent audit
      CLAIMED  — supplier self-declaration without independent verification
      MISSING  — entity or data not found in sources
      MIXED    — some verified, some only claimed

    If the queried entity does not appear in the sources, set evidence_level=MISSING
    and explain in the answer field.
""").strip()

# ---------------------------------------------------------------------------
# Pydantic schemas
# ---------------------------------------------------------------------------


class EvidenceLevel(str, Enum):
    VERIFIED = "VERIFIED"
    CLAIMED = "CLAIMED"
    MISSING = "MISSING"
    MIXED = "MIXED"


class RiskLevel(str, Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"


class StructuredRAGResponse(BaseModel):
    answer: str = Field(description="Main answer in plain language")
    evidence_level: EvidenceLevel = Field(description="Overall evidence quality")
    sources_used: list[str] = Field(default_factory=list)
    gaps: list[str] = Field(default_factory=list)
    risk: RiskLevel = Field(default=RiskLevel.MEDIUM)

    def pretty_print(self) -> None:
        icon = {"VERIFIED": "✓", "CLAIMED": "~", "MISSING": "✗", "MIXED": "±"}.get(
            self.evidence_level, "?"
        )
        risk_icon = {"LOW": "🟢", "MEDIUM": "🟡", "HIGH": "🔴"}.get(self.risk, "")
        print(f"{icon} Evidence: {self.evidence_level}   {risk_icon} Risk: {self.risk}")
        print(f"Answer: {self.answer}")
        if self.sources_used:
            print(f"Sources: {', '.join(self.sources_used)}")
        if self.gaps:
            print("Gaps:")
            for g in self.gaps:
                print(f"  · {g}")


def parse_structured_response(text: str) -> StructuredRAGResponse | None:
    """Extract and parse the first JSON object from LLM output. Returns None on failure."""
    json_match = re.search(r"\{.*\}", text, re.DOTALL)
    if not json_match:
        return None
    try:
        data = json.loads(json_match.group())
        return StructuredRAGResponse(**data)
    except Exception:
        return None


def build_agent(
    vector_store,
    embedding_model,
    llm,
    system_prompt: str = SYSTEM_PROMPT_ENTITY_GROUNDED,
    top_k: int = 5,
) -> RAG:
    """Build a RAG agent with the specified system prompt."""
    retriever = VectorStoreRetriever(embedding_model, vector_store, top_k=top_k)
    return RAG(
        llm=llm,
        utility_llm=llm,
        system_prompt=system_prompt,
        retrievers=[retriever],
        number_query_expansion=0,
    )


# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------

# Choose your LLM backend: "ollama" or "openai"
BACKEND = "openai"  # set this before running

if not BACKEND:
    raise ValueError(
        'BACKEND is not set. Edit the line above and set it to "ollama" or "openai".'
    )

embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)
chunks = load_chunks()
vector_store = await build_vector_store(
    chunks, embedding_model, db_path=VS_PATH, reset=False
)

llm = build_llm(backend=BACKEND)

print(f"Vector store : {VS_PATH}")
print(f"LLM backend  : {BACKEND}")
print("Setup complete.")

---

## The Problem: Unstructured Answers Obscure Claim Quality

The baseline RAG answers in plain prose. Run the same GWP question on two products with very different evidence status and read both answers. Can you tell from the text alone which figure is independently verified and which one is a supplier self-declaration?

In [2]:
baseline_agent = build_agent(
    vector_store, embedding_model, llm, system_prompt=SYSTEM_PROMPT_BASELINE, top_k=5
)

queries = [
    # Logypal 1: third-party verified EPD under ISO 14044
    "What is the carbon footprint of the Logypal 1 pallet, and is it verified?",
    # tesa ECO tape: internal comparative assessment, no independent EPD
    "How much lower is the carbon footprint of tesa ECO tape compared to standard tape?",
]

for q in queries:
    resp = await baseline_agent.answer(QueryWithContext(query=q, history=[]))
    print("-" * 70)
    print(f"Q: {q}")
    print("-" * 70)
    print(f"A: {resp.content}")
    print()

2026-02-25 09:41:21.658 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:get_embeddings:76 - sentence-transformers/all-MiniLM-L6-v2 embeddings size: (1, 384)
2026-02-25 09:41:23.917 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:get_embeddings:76 - sentence-transformers/all-MiniLM-L6-v2 embeddings size: (1, 384)


----------------------------------------------------------------------
Q: What is the carbon footprint of the Logypal 1 pallet, and is it verified?
----------------------------------------------------------------------
A: The carbon footprint of the Logypal 1 pallet is reported as 4.1 kg CO₂e per pallet based on an internal, non-verified LCA from 2021, but a third-party verified EPD published in 2023 provides updated figures that should be referenced instead; however, the exact carbon footprint from the EPD is not specified in the sources.

----------------------------------------------------------------------
Q: How much lower is the carbon footprint of tesa ECO tape compared to standard tape?
----------------------------------------------------------------------
A: The carbon footprint of tesa ECO tape is reported to have a 68% reduction in CO₂ emissions compared to the conventional tesapack baseline product from 2019, but this assessment has not been independently verified.



---

## Prompt-Based JSON Coercion

The simplest approach: add JSON formatting instructions to the system prompt. `SYSTEM_PROMPT_STRUCTURED` instructs the LLM to always respond with a JSON object matching the evidence schema.

`parse_structured_response()` handles extraction with a `re.search(r"\{.*\}", ...)` regex, which tolerates Markdown code fences and leading prose. But it can fail on badly malformed output.

In [5]:
print("STRUCTURED SYSTEM PROMPT (defines the JSON schema via prose):")
print("=" * 60)
print(SYSTEM_PROMPT_STRUCTURED)

STRUCTURED SYSTEM PROMPT (defines the JSON schema via prose):
You are a sustainability compliance assistant for PrimePack AG.
Answer questions using ONLY the provided sources.

Always respond with a JSON object matching this schema:
{
  "answer": "<plain language answer>",
  "evidence_level": "<VERIFIED | CLAIMED | MISSING | MIXED>",
  "sources_used": ["<source file 1>", ...],
  "gaps": ["<missing or unclear information>", ...],
  "risk": "<LOW | MEDIUM | HIGH — risk of citing this in customer communication>"
}

Evidence levels:
  VERIFIED — third-party EPD or independent audit
  CLAIMED  — supplier self-declaration without independent verification
  MISSING  — entity or data not found in sources
  MIXED    — some verified, some only claimed

If the queried entity does not appear in the sources, set evidence_level=MISSING
and explain in the answer field.


In [5]:
CLAIM_QUERY = (
    "Is the carbon footprint claim for the tesa ECO tape independently verified?"
)

structured_agent = build_agent(
    vector_store, embedding_model, llm, system_prompt=SYSTEM_PROMPT_STRUCTURED, top_k=5
)
resp = await structured_agent.answer(QueryWithContext(query=CLAIM_QUERY, history=[]))

print("Raw LLM output (prompt-based coercion):")
print("-" * 60)
print(resp.content)

2026-02-25 09:42:51.381 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:get_embeddings:76 - sentence-transformers/all-MiniLM-L6-v2 embeddings size: (1, 384)


Raw LLM output (prompt-based coercion):
------------------------------------------------------------
{
  "answer": "The carbon footprint claim for the tesa ECO tape is not independently verified. It is based on an internal assessment by Tesa and does not have a verified Environmental Product Declaration (EPD).",
  "evidence_level": "CLAIMED",
  "sources_used": ["1151593a-3b54-4853-8684-f1579b019223", "418cd1bb-d289-469b-ba33-d2338a18b054"],
  "gaps": ["No independent verification or EPD available for the carbon footprint claim."],
  "risk": "HIGH"
}


In [6]:
# parse_structured_response: re.search for JSON block -> json.loads -> Pydantic validation
parsed = parse_structured_response(resp.content)

if parsed:
    print("Parsed successfully:")
    parsed.pretty_print()
else:
    print("[Parse failed — no valid JSON block found in the output above]")
    print("This is the key weakness of prompt-only coercion.")

Parsed successfully:
~ Evidence: EvidenceLevel.CLAIMED   🔴 Risk: RiskLevel.HIGH
Answer: The carbon footprint claim for the tesa ECO tape is not independently verified. It is based on an internal assessment by Tesa and does not have a verified Environmental Product Declaration (EPD).
Sources: 1151593a-3b54-4853-8684-f1579b019223, 418cd1bb-d289-469b-ba33-d2338a18b054
Gaps:
  · No independent verification or EPD available for the carbon footprint claim.


> **Observation:** When it works, the structured output immediately surfaces the claim quality: `CLAIMED`, `risk: HIGH`, with a gaps list noting the absence of an independent EPD. When the LLM adds a preamble or Markdown fences, the regex may find no JSON block and return `None`. The next section eliminates this failure mode.

---

## OpenAI JSON Mode (Guaranteed Valid JSON)

OpenAI's JSON mode (`response_format={"type": "json_object"}`) guarantees that the model's output is always syntactically valid JSON, no Markdown fences, no preamble, no `json.loads` exceptions. The schema is still defined by the system prompt; the API just enforces parseability.

**Requirements:**
- The system prompt must mention the word "JSON" (otherwise the API raises an error)
- Requires an OpenAI model (`gpt-4o-mini`, `gpt-4o`, etc.) -> not available with Ollama

`OpenAILLM` already accepts `response_format` as a constructor argument, one line changes.

In [7]:
if BACKEND == "openai":
    # Instantiate with JSON mode enabled
    llm_json_mode = OpenAILLM(
        model_name="gpt-4o-mini",
        temperature=0.3,
        response_format={"type": "json_object"},
    )
    json_agent = build_agent(
        vector_store,
        embedding_model,
        llm_json_mode,
        system_prompt=SYSTEM_PROMPT_STRUCTURED,
        top_k=5,
    )
    resp_json = await json_agent.answer(QueryWithContext(query=CLAIM_QUERY, history=[]))

    print("Raw LLM output (OpenAI JSON mode — API-guaranteed parseable):")
    print("-" * 60)
    print(resp_json.content)
    print()

    # json.loads never fails here — the API guarantees valid JSON
    structured_json = StructuredRAGResponse(**json.loads(resp_json.content))
    print("Parsed result:")
    structured_json.pretty_print()
else:
    print("JSON mode requires the OpenAI backend.")
    print(
        "With Ollama, use Approach A and wrap parse_structured_response() in a try/except."
    )

2026-02-25 09:44:32.238 | DEBUG    | conversational_toolkit.llms.openai:__init__:63 - OpenAI LLM loaded: gpt-4o-mini; temperature: 0.3; seed: 42; tools: None; tool_choice: None; response_format: {'type': 'json_object'}
2026-02-25 09:44:32.457 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:get_embeddings:76 - sentence-transformers/all-MiniLM-L6-v2 embeddings size: (1, 384)


Raw LLM output (OpenAI JSON mode — API-guaranteed parseable):
------------------------------------------------------------
{
  "answer": "The carbon footprint claim for the tesa ECO tape is not independently verified. It is based on an internal Tesa assessment, which is classified as unverified according to PrimePack AG's internal policy.",
  "evidence_level": "CLAIMED",
  "sources_used": ["ART_internal_procurement_policy.md", "ART_supplier_brochure_tesa_ECO.md", "Response to Section 1: Tape Products"],
  "gaps": ["No independent verification or EPD available for the tesa ECO tape."],
  "risk": "MEDIUM"
}

Parsed result:
~ Evidence: EvidenceLevel.CLAIMED   🟡 Risk: RiskLevel.MEDIUM
Answer: The carbon footprint claim for the tesa ECO tape is not independently verified. It is based on an internal Tesa assessment, which is classified as unverified according to PrimePack AG's internal policy.
Sources: ART_internal_procurement_policy.md, ART_supplier_brochure_tesa_ECO.md, Response to Section

> **Observation:** The raw output is a clean JSON object every time. `json.loads()` never raises an exception. The only remaining failure mode is at the field level: the LLM might produce an `evidence_level` value that is not a valid enum (e.g. `"UNVERIFIED"` instead of `"CLAIMED"`).

---

## 5. Approach C: Pydantic Validation Layer

Pydantic validates the parsed JSON against the typed schema:

```python
class EvidenceLevel(str, Enum):
    VERIFIED = "VERIFIED"
    CLAIMED  = "CLAIMED"
    MISSING  = "MISSING"
    MIXED    = "MIXED"

class StructuredRAGResponse(BaseModel):
    answer:         str
    evidence_level: EvidenceLevel   # must be one of the four values above
    sources_used:   list[str]
    gaps:           list[str]
    risk:           RiskLevel       # LOW / MEDIUM / HIGH
```

A `ValidationError` is raised if the LLM returns an unknown enum value, a missing required field, or the wrong type. This makes the structured output safe to pass into downstream code without silent corruption.

Combine with JSON mode for maximum reliability: JSON mode guarantees syntactically valid JSON; Pydantic guarantees semantically valid structure.

In [ ]:
# Simulate three possible LLM outputs and show Pydantic's behaviour on each
malformed_examples = [
    # Wrong enum value
    '{"answer": "...", "evidence_level": "UNVERIFIED", "sources_used": [], "gaps": [], "risk": "HIGH"}',
    # Missing required field (risk)
    '{"answer": "...", "evidence_level": "CLAIMED", "sources_used": [], "gaps": []}',
    # Correct
    '{"answer": "The claim is unverified.", "evidence_level": "CLAIMED", "sources_used": ["ART_supplier_brochure_tesa_ECO.pdf"], "gaps": ["No independent EPD"], "risk": "HIGH"}',
]

print("Pydantic validation on three JSON examples:\n")
for i, raw in enumerate(malformed_examples, 1):
    try:
        result = StructuredRAGResponse(**json.loads(raw))
        print(f"Example {i}: VALID")
        result.pretty_print()
    except ValidationError as e:
        print(f"Example {i}: INVALID — {e.error_count()} error(s)")
        for err in e.errors():
            print(f"  field={err['loc']}  type={err['type']}  msg={err['msg']}")
    print()

> **Observation:** Pydantic catches both the wrong enum value and the missing field that would otherwise pass silently as `None`. In a production pipeline, a `ValidationError` on a single query can trigger a retry with a corrective prompt rather than propagating a corrupted record into the compliance database.

---

## 6. Verified vs Claimed: Side-by-Side

Two products in the portfolio have very different evidence status:

- **Logypal 1 (32-103):** third-party verified EPD under ISO 14044 → should produce `VERIFIED`, `risk: LOW`
- **CPR Wooden Pallet (32-101):** internal LCA with no independent EPD → should produce `CLAIMED`, `risk: MEDIUM` or `HIGH`

The structured output makes this difference machine-readable.

In [ ]:
products = [
    ("Logypal 1 pallet (product ID 32-103)", "third-party verified EPD expected"),
    ("CPR Wooden Pallet 1208 (product ID 32-101)", "internal LCA only expected"),
]

evidence_agent = build_agent(
    vector_store,
    embedding_model,
    llm_json_mode if BACKEND == "openai" else llm,
    system_prompt=SYSTEM_PROMPT_STRUCTURED,
    top_k=5,
)

for product, expectation in products:
    query = f"What is the Global Warming Potential (GWP) of the {product}?"
    resp = await evidence_agent.answer(QueryWithContext(query=query, history=[]))

    print(f"\nProduct  : {product}")
    print(f"Expected : {expectation}")
    print("-" * 50)

    if BACKEND == "openai":
        try:
            parsed = StructuredRAGResponse(**json.loads(resp.content))
            parsed.pretty_print()
        except (json.JSONDecodeError, ValidationError) as e:
            print(f"Parse error: {e}")
    else:
        fallback = parse_structured_response(resp.content)
        if fallback:
            fallback.pretty_print()
        else:
            print(resp.content[:400])

> **Observation:** The Logypal 1 should come back `VERIFIED` / `risk: LOW`; the CPR Wooden Pallet should come back `CLAIMED` / `risk: MEDIUM` or `HIGH`. A compliance workflow can now filter or flag all `CLAIMED` answers automatically - no manual review of every prose response.

---

## 7. Multi-Product Comparison Table

Structured outputs are not limited to single-question answers. This section asks the LLM to return a **portfolio-level comparison** as a typed list, one entry per pallet. This pattern generates audit summaries, compliance dashboards, or CSRD data tables without any manual copy-paste.

We define a new Pydantic schema and use a higher `top_k` to gather context from multiple product documents in one call.

In [ ]:
class ProductGWP(BaseModel):
    product_name: str = Field(
        description="Full product name as it appears in the source"
    )
    product_id: str = Field(
        default="", description="Product ID if available, else empty string"
    )
    gwp_value: str = Field(description="GWP figure with unit, or 'not available'")
    evidence_level: EvidenceLevel
    source_document: str = Field(description="Source file name")


class PortfolioGWPComparison(BaseModel):
    products: list[ProductGWP]
    summary: str = Field(
        description="One-sentence overall assessment of portfolio evidence quality"
    )


print("Schema: PortfolioGWPComparison")
print(f"  products : list of ProductGWP ({len(ProductGWP.model_fields)} fields each)")
for name, field in ProductGWP.model_fields.items():
    print(f"    {name:<20} {field.description or ''}")
print("  summary  : str")

In [ ]:
COMPARISON_SYSTEM_PROMPT = """
You are a sustainability compliance assistant for PrimePack AG.
Answer using ONLY the provided sources. Do not invent data.

Always respond with a JSON object matching this exact schema:
{
  "products": [
    {
      "product_name": "<full name from source>",
      "product_id": "<product ID or empty string>",
      "gwp_value": "<GWP with unit, or 'not available'>",
      "evidence_level": "<VERIFIED | CLAIMED | MISSING>",
      "source_document": "<source file name>"
    }
  ],
  "summary": "<one-sentence overall assessment>"
}

Evidence levels:
  VERIFIED - backed by a third-party EPD or independent audit
  CLAIMED  - supplier self-declaration, not independently verified
  MISSING  - GWP data not found in the provided sources
""".strip()

COMPARISON_QUERY = (
    "List the Global Warming Potential (GWP) for all pallet products in the portfolio. "
    "For each product provide the GWP value, evidence level, and source document."
)

print(f"Query: {COMPARISON_QUERY!r}")

In [ ]:
# top_k=10 to gather context across multiple product documents in one call
if BACKEND == "openai":
    llm_comparison = OpenAILLM(
        model_name="gpt-4o-mini",
        temperature=0.1,  # low temperature for factual extraction
        response_format={"type": "json_object"},
    )
else:
    llm_comparison = llm

comparison_agent = build_agent(
    vector_store,
    embedding_model,
    llm_comparison,
    system_prompt=COMPARISON_SYSTEM_PROMPT,
    top_k=10,
)
resp_comparison = await comparison_agent.answer(
    QueryWithContext(query=COMPARISON_QUERY, history=[])
)

print("Raw LLM output:")
print("-" * 60)
print(resp_comparison.content)

In [ ]:
try:
    if BACKEND == "openai":
        raw_data = json.loads(resp_comparison.content)
    else:
        match = re.search(r"\{.*\}", resp_comparison.content, re.DOTALL)
        raw_data = json.loads(match.group()) if match else None
        if raw_data is None:
            raise ValueError("No JSON block found in output")
    comparison = PortfolioGWPComparison(**raw_data)
except Exception as e:
    print(f"Parse failed: {e}")
    comparison = None

if comparison:
    print(f"{'Product':<40}  {'ID':<10}  {'Evidence':>10}  GWP")
    print("-" * 90)
    for p in comparison.products:
        icon = {"VERIFIED": "✓", "CLAIMED": "~", "MISSING": "✗"}.get(
            p.evidence_level.value, "?"
        )
        print(
            f"{p.product_name:<40}  {p.product_id:<10}  {icon} {p.evidence_level.value:>9}  {p.gwp_value}"
        )
    print()
    print(f"Summary: {comparison.summary}")
    verified = sum(
        1 for p in comparison.products if p.evidence_level == EvidenceLevel.VERIFIED
    )
    print(f"\nVerified (third-party EPD): {verified} / {len(comparison.products)}")

> **Observation:** A single LLM call returns a structured table covering the entire pallet portfolio. Each row carries a machine-readable evidence label. This output can be inserted directly into a CSRD reporting template, a compliance database, or a dashboard without manual copy-paste. The `MISSING` rows tell you exactly where data collection gaps exist.

---

## Summary

| Approach | Reliability | When to use |
|---|---|---|
| Entity grounding prompt | Prompt-dependent | Always - necessary foundation; prevents hallucinated `VERIFIED` labels |
| Prompt-based JSON (A) | Low - output format not enforced | Ollama / any backend; wrap in try/except |
| OpenAI JSON mode (B) | High - syntactically guaranteed | OpenAI models; one constructor argument |
| Pydantic validation (C) | Catches field-level errors | Combine with B for production; add retry on `ValidationError` |
| Multi-field comparison | Depends on retrieval quality | Audit tables, CSRD summaries, compliance dashboards |

**Recommended production stack:** entity-grounding prompt + OpenAI JSON mode + Pydantic validation. For Ollama: entity-grounding prompt + `parse_structured_response()` with retry on `None`.

**Next: Feature Track 4:** Improve retrieval quality with score thresholds, contextual enrichment, query expansion, HyDE, BM25, hybrid search, and LLM reranking.

---

## Reference Queries: Structured Output Progress

Structured outputs add machine-readable quality signals to answers. Run the same queries as in `feature0a_baseline_rag.ipynb` with `evidence_agent` and compare:

| Query | Baseline prose | Structured output |
|---|---|---|
| "What is the GWP of the Logypal 1 pallet?" | A sentence with the GWP value, no evidence label | `evidence_level: VERIFIED`, `risk: LOW`, source citation |
| "Is any product FSC-C147829 certified?" | A yes/no statement in prose | `claim_supported: true`, exact quote from the source |
| "What carbon neutrality claims does tesa make?" | A paragraph mixing claims and context | Separate fields for claim, evidence level, and caveats |